# Student Performance Prediction using Random Forest Classifier

**Dataset:** ExamScore Dataset (20,000 student records)  
**Algorithm:** Random Forest Classifier (Entropy, max_depth=5)  
**Goal:** Predict whether a student will pass or fail based on study behaviour and academic factors.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('ExamScore.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Exam score distribution
plt.figure(figsize=(8, 4))
plt.hist(df['exam_score'], bins=30, color='steelblue', edgecolor='black')
plt.axvline(x=40, color='tomato', linestyle='--', linewidth=2, label='Pass threshold (40)')
plt.title('Exam Score Distribution')
plt.xlabel('Exam Score')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Study hours vs exam score
plt.figure(figsize=(8, 4))
plt.scatter(df['study_hours'], df['exam_score'], alpha=0.3, color='steelblue', s=10)
plt.axhline(y=40, color='tomato', linestyle='--', linewidth=1.5, label='Pass threshold')
plt.title('Study Hours vs Exam Score')
plt.xlabel('Study Hours')
plt.ylabel('Exam Score')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Average exam score by study method
avg_by_method = df.groupby('study_method')['exam_score'].mean().sort_values(ascending=False)
avg_by_method.plot(kind='bar', color='steelblue', edgecolor='black', figsize=(8, 4))
plt.title('Average Exam Score by Study Method')
plt.xlabel('Study Method')
plt.ylabel('Average Exam Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Average exam score by exam difficulty
avg_by_difficulty = df.groupby('exam_difficulty')['exam_score'].mean().sort_values(ascending=False)
avg_by_difficulty.plot(kind='bar', color='tomato', edgecolor='black', figsize=(6, 4))
plt.title('Average Exam Score by Exam Difficulty')
plt.xlabel('Exam Difficulty')
plt.ylabel('Average Exam Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numerical features only)
plt.figure(figsize=(8, 5))
numeric_cols = df.select_dtypes(include=np.number).drop(columns=['student_id'])
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Correlation Heatmap (Numerical Features)')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Create binary target: 1 = Pass (score >= 40), 0 = Fail
# Passing threshold is set at 40 marks — standard academic pass mark
df['Result'] = np.where(df['exam_score'] >= 40, 1, 0)

print("Class distribution (0 = Fail, 1 = Pass):")
print(df['Result'].value_counts())

df['Result'].value_counts().plot(kind='bar', color=['tomato', 'steelblue'], edgecolor='black')
plt.title('Pass / Fail Distribution')
plt.xlabel('Result (0 = Fail, 1 = Pass)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Select features focused on study behaviour
# Excluded: student_id (identifier), age/gender/course (demographic — not study behaviour)
feature_cols = [
    'study_hours', 'class_attendance', 'internet_access', 'sleep_hours',
    'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty'
]

X = df[feature_cols]
y = df['Result']

# Encode categorical features
categorical_features = X.select_dtypes(include='object').columns
print("Categorical features encoded:", list(categorical_features))
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)
print("Encoded feature shape:", X_encoded.shape)
X_encoded.head(3)

In [ ]:
# Train-test split (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.3, random_state=1, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

## 5. Model Training

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,     # 100 decision trees
    criterion='entropy',  # information gain for splits
    max_depth=5,          # limit depth to prevent overfitting
    max_features='sqrt',  # number of features considered at each split
    random_state=1
)
rf.fit(X_train, y_train)
print("Model trained successfully.")

## 6. Model Evaluation

In [ ]:
y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Fail', 'Pass']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fail', 'Pass'],
            yticklabels=['Fail', 'Pass'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importance = pd.Series(
    rf.feature_importances_, index=X_encoded.columns
).sort_values(ascending=True)

importance.plot(kind='barh', color='steelblue', edgecolor='black', figsize=(9, 6))
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 7. Summary

| Item | Detail |
|------|--------|
| Dataset | ExamScore Dataset (20,000 student records, 13 columns) |
| Algorithm | Random Forest Classifier (Entropy, max_depth=5, 100 trees) |
| Target | Pass (score ≥ 40) / Fail (score < 40) |
| Features Used | 8 study behaviour features |
| Encoding | One-hot encoding for categorical features |
| Train/Test Split | 70% / 30% (stratified) |

**Key Observations:**
- Study hours and class attendance are the strongest predictors of exam outcome.
- Random Forest outperforms a single Decision Tree by combining 100 trees to reduce overfitting.
- Demographic features (age, gender, course) were intentionally excluded to focus on study behaviour.